# Video 6a: Cell Painting - From Images to Morphological Features
### AI for Drug Discovery Series - DigitalSreeni

In this notebook we work with real Cell Painting images from the JUMP-CP pilot dataset published by the Broad Institute. We go through the full pipeline that industrial HCA labs run on every compound plate:

1. Load raw 16-bit fluorescence images from the five Cell Painting channels
2. Visualize each channel and build a false-color composite
3. Segment nuclei and whole cells using Cellpose
4. Extract morphological features from each segmented object - the same families of features that CellProfiler computes at scale across thousands of wells
5. Display and interpret the features

In Video 6b we skip directly to step 4 output - loading 10,752 pre-computed well-level profiles from 1,571 compounds - and use PCA, UMAP, and hierarchical clustering to ask whether compounds with the same mechanism of action produce similar morphological fingerprints.

**Dataset**: JUMP-CP pilot (cpg0000), Broad Institute. CC0 license.  
**Cell line**: U2OS human osteosarcoma cells, 48-hour compound treatment.  
**Image size**: 1080 x 1080 pixels, 16-bit grayscale per channel, 20x magnification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import tifffile
import urllib.request
import io
import os
from pathlib import Path
from scipy import ndimage
from scipy.stats import skew, kurtosis
from skimage.measure import regionprops_table
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'Arial',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150
})

# Create results directory for all output figures and CSVs
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

print('Libraries loaded.')
print(f'Results will be saved to: {RESULTS_DIR.resolve()}')

## Part 1: Load Cell Painting Images

The JUMP-CP pilot dataset is hosted on AWS S3 as a public open-data resource - no credentials needed. Each well site produces five fluorescence TIFFs (one per stain channel) plus brightfield images. We download and cache them locally so re-running the notebook is fast.

The images are 16-bit grayscale. Pixel values range from 0 to 65535, representing photon counts from the camera detector. The 16-bit depth preserves fine intensity differences that would be lost at 8-bit, and those differences are exactly what feature extraction relies on.

**Channel assignments for JUMP-CP:**
- ch1 = DNA (Hoechst 33342) - nucleus
- ch2 = ER (Concanavalin A) - endoplasmic reticulum
- ch3 = RNA (SYTO 14) - nucleoli and cytoplasmic RNA
- ch4 = AGP (Phalloidin + WGA) - actin, Golgi, plasma membrane
- ch5 = Mito (MitoTracker) - mitochondria

In [ ]:
# S3 base path - public Cell Painting Gallery, no authentication required
S3_BASE = 'https://cellpainting-gallery.s3.amazonaws.com/cpg0000-jump-pilot/source_4/images'
BATCH = '2020_11_04_CPJUMP1'

# Two plates: one DMSO control well and one compound-treated well
# Both from plate BR00117010 - same plate, different wells
PLATE = 'BR00117010__2020-11-08T18_18_00-Measurement1'

# JUMP-CP cpg0000 actual channel order (confirmed from Pfizer moa-profiler repo and Broad docs):
# ch1=Mito (Alexa 647), ch2=AGP (Alexa 568), ch3=RNA (Alexa 488), ch4=ER (Alexa 488), ch5=DNA (Hoechst)
CHANNEL_NAMES  = ['Mito', 'AGP', 'RNA', 'ER', 'DNA']
CHANNEL_LABELS = [
    'Ch1: Mito (Alexa 647)',
    'Ch2: AGP (Phalloidin + WGA)',
    'Ch3: RNA (SYTO 14)',
    'Ch4: ER (Concanavalin A)',
    'Ch5: DNA (Hoechst 33342)'
]
# False-color display colors matching Cell Painting convention
CHANNEL_COLORS = [
    np.array([0.8, 0.2, 0.9]),   # ch1 Mito - magenta
    np.array([1.0, 0.2, 0.2]),   # ch2 AGP  - red
    np.array([1.0, 0.9, 0.0]),   # ch3 RNA  - yellow
    np.array([0.2, 0.8, 0.2]),   # ch4 ER   - green
    np.array([0.2, 0.4, 1.0]),   # ch5 DNA  - blue
]

# Local cache directory - images are ~2 MB each so we only download once
CACHE_DIR = Path('cell_painting_images')
CACHE_DIR.mkdir(exist_ok=True)

def load_channel(batch, plate, well_row, well_col, field, channel, cache_dir=CACHE_DIR):
    """Load a single channel TIFF from S3, using local cache if available.

    Parameters
    ----------
    well_row, well_col : int - 1-indexed row and column of the well
    field : int - imaging site within the well (1-indexed)
    channel : int - fluorescence channel 1-5

    Returns
    -------
    numpy array, shape (H, W), dtype uint16
    """
    filename = f'r{well_row:02d}c{well_col:02d}f{field:02d}p01-ch{channel}sk1fk1fl1.tiff'
    cache_path = cache_dir / f'{plate[:12]}_{filename}'

    if cache_path.exists():
        return tifffile.imread(str(cache_path))

    url = f'{S3_BASE}/{batch}/images/{plate}/Images/{filename}'
    print(f'  Downloading {filename} ...', end=' ')
    with urllib.request.urlopen(url, timeout=60) as r:
        data = r.read()
    img = tifffile.imread(io.BytesIO(data))
    tifffile.imwrite(str(cache_path), img)
    print(f'done ({img.nbytes / 1024:.0f} KB)')
    return img

# Load two wells: r01c01 (DMSO control) and r03c03 (compound treated)
print('Loading DMSO control well (r01c01)...')
channels_dmso = [load_channel(BATCH, PLATE, 1, 1, 1, ch) for ch in range(1, 6)]

print('Loading compound-treated well (r03c03)...')
channels_treated = [load_channel(BATCH, PLATE, 3, 3, 1, ch) for ch in range(1, 6)]

H, W = channels_dmso[0].shape
print(f'\nImage dimensions: {H} x {W} pixels, 16-bit, {len(channels_dmso)} channels per well')

## Part 2: Visualize Each Channel

Each channel is displayed with a colormap matching the biological stain color. Images are contrast-stretched for display by clipping to the 1st-99th percentile of pixel values - this removes background noise and a few saturated pixels without changing the underlying data.

The contrast stretch is for display only. All feature extraction operates on raw 16-bit values.

In [ ]:
def contrast_stretch(img, low_pct=1, high_pct=99):
    """Clip to percentile range and rescale to 0-1 for display."""
    lo = np.percentile(img, low_pct)
    hi = np.percentile(img, high_pct)
    out = np.clip(img, lo, hi).astype(float)
    return (out - lo) / (hi - lo + 1e-8)

channel_cmaps = ['PuRd', 'Reds', 'Wistia', 'Greens', 'Blues']   # Mito, AGP, RNA, ER, DNA

fig, axes = plt.subplots(2, 5, figsize=(18, 7.5))

for i, (label, cmap) in enumerate(zip(CHANNEL_LABELS, channel_cmaps)):
    axes[0, i].imshow(contrast_stretch(channels_dmso[i]), cmap=cmap)
    axes[0, i].set_title(label, fontsize=8.5, pad=4)
    axes[0, i].axis('off')

    axes[1, i].imshow(contrast_stretch(channels_treated[i]), cmap=cmap)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('DMSO control\n(r01c01)', fontsize=10, labelpad=6)
axes[1, 0].set_ylabel('Compound treated\n(r03c03)', fontsize=10, labelpad=6)
for ax in [axes[0, 0], axes[1, 0]]:
    ax.yaxis.set_visible(True)
    ax.set_yticks([])

fig.suptitle(
    'Cell Painting - Five Channels, Two Wells\n'
    'U2OS cells, JUMP-CP pilot dataset (cpg0000), Broad Institute',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig1_five_channels.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig1_five_channels.png')

## Part 3: False-Color Composite

In Cell Painting publications, the five channels are merged into a single RGB composite using conventional false colors. Each stain is assigned a color and the contributions are summed:

- DNA - blue
- ER - green
- RNA - yellow (red + green)
- AGP - red
- Mito - magenta (red + blue)

This is the image you see on the cover of Cell Painting papers. No microscope captures this directly - it is always a computational composite.

In [ ]:
def make_composite(channels):
    """Build standard Cell Painting false-color RGB composite."""
    mito, agp, rna, er, dna = [contrast_stretch(c) for c in channels]   # ch1=Mito, ch2=AGP, ch3=RNA, ch4=ER, ch5=DNA
    R = np.clip(agp + rna * 0.8 + mito * 0.6, 0, 1)
    G = np.clip(er  + rna * 0.8, 0, 1)
    B = np.clip(dna + mito * 0.6, 0, 1)
    composite = np.stack([R, G, B], axis=-1)
    composite = composite / (composite.max() + 1e-8)
    return np.clip(composite, 0, 1)

comp_dmso    = make_composite(channels_dmso)
comp_treated = make_composite(channels_treated)

# Build color legend patches
legend_items = [
    ('DNA  (Hoechst)',     [0.3, 0.5, 1.0]),
    ('ER   (ConA)',        [0.2, 0.8, 0.2]),
    ('RNA  (SYTO 14)',     [1.0, 1.0, 0.0]),
    ('AGP  (Phalloidin)',  [1.0, 0.3, 0.3]),
    ('Mito (MitoTracker)', [0.9, 0.3, 0.9]),
]

fig = plt.figure(figsize=(16, 6))
gs = gridspec.GridSpec(1, 3, width_ratios=[4, 4, 1.2], figure=fig)

ax0 = fig.add_subplot(gs[0])
ax0.imshow(comp_dmso)
ax0.set_title('DMSO control (r01c01)', fontsize=11)
ax0.axis('off')

ax1 = fig.add_subplot(gs[1])
ax1.imshow(comp_treated)
ax1.set_title('Compound treated (r03c03)', fontsize=11)
ax1.axis('off')

ax2 = fig.add_subplot(gs[2])
ax2.set_facecolor('black')
ax2.set_title('Channel colors', fontsize=10, color='white', pad=8)
patches = [mpatches.Patch(color=c, label=l) for l, c in legend_items]
ax2.legend(handles=patches, loc='center', frameon=False,
           labelcolor='white', fontsize=9, handlelength=1.5)
ax2.set_xticks([])
ax2.set_yticks([])
for spine in ax2.spines.values():
    spine.set_visible(False)

fig.suptitle('Cell Painting - False-Color Composite (5 channels merged)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig2_composite.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig2_composite.png')

## Part 4: Zoom In to See Individual Cells

The full 1080x1080 field contains hundreds of cells. We crop a 400x400 pixel region to see individual cells clearly. At 20x magnification, one pixel is approximately 0.65 micrometers, so 400 pixels spans about 260 micrometers - enough to see 30-50 cells in detail.

In this zoomed view you can see:
- Bright blue blobs: individual nuclei
- Red/orange web around nuclei: actin cytoskeleton defining cell boundaries
- Magenta dots scattered through cytoplasm: mitochondria
- Green reticulated network: endoplasmic reticulum

Each of these structures will be measured quantitatively in Part 5.

In [ ]:
# Crop region - adjust these coordinates to find a well-populated area
CY, CX, CS = 200, 200, 420   # top-left y, top-left x, crop size in pixels

crop_dmso    = comp_dmso[CY:CY+CS, CX:CX+CS]
crop_treated = comp_treated[CY:CY+CS, CX:CX+CS]

# Also crop each raw channel for feature extraction later
ch_crop_dmso = [ch[CY:CY+CS, CX:CX+CS] for ch in channels_dmso]

# Scale bar: at 20x magnification, ~6.5 pixels per micron
# 100 um = ~65 pixels
scale_px = 65

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, crop, label in zip(axes, [crop_dmso, crop_treated],
                            ['DMSO control - zoomed', 'Compound treated - zoomed']):
    ax.imshow(crop)
    ax.set_title(label, fontsize=11)
    ax.axis('off')
    # Scale bar
    bar_x, bar_y = 18, CS - 28
    ax.plot([bar_x, bar_x + scale_px], [bar_y, bar_y], color='white', linewidth=2.5)
    ax.text(bar_x + scale_px / 2, bar_y - 10, '100 um',
            color='white', ha='center', fontsize=9)

fig.suptitle(
    'Cell Painting - Zoomed View\n'
    'Blue = DNA (nuclei), Red = actin, Green = ER, Magenta = mitochondria',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig3_zoomed.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig3_zoomed.png')

## Part 5: Cell Segmentation with Cellpose

Before we can measure anything per-cell, we need to identify where each cell and nucleus is in the image. This is the segmentation step.

We use Cellpose, a deep learning segmentation tool trained on a large diverse collection of fluorescence microscopy images. It handles crowded, touching cells reliably without requiring manual parameter tuning for each new image type.

**Two-pass segmentation:**
1. **Nuclei model** on the DNA channel - finds the bright, compact nucleus of each cell
2. **cyto3 model** on the AGP channel (actin) guided by the DNA channel - expands from each nucleus outward to find the full cell boundary

The output is a labeled mask array where each pixel carries the integer ID of the cell it belongs to (0 = background).

In [ ]:
from cellpose import models
import cellpose
print(f'Cellpose version: {cellpose.__version__}')

# Correct channel indices for JUMP-CP cpg0000:
# ch5 (index 4) = DNA (Hoechst 33342) - nuclei
# ch4 (index 3) = ER (Concanavalin A) - fills cytoplasm uniformly, best for whole-cell boundary
dna_crop = ch_crop_dmso[4]   # ch5 = DNA
er_crop  = ch_crop_dmso[3]   # ch4 = ER

# --- Nucleus segmentation ---
# Pass DNA channel directly as a single grayscale image
# Cellpose v4+ does not require the channels= parameter
print('Running Cellpose nuclei model on DNA channel (ch5)...')
nuc_model = models.CellposeModel(model_type='nuclei', gpu=False)
nuc_masks, nuc_flows, _ = nuc_model.eval(
    dna_crop,
    diameter=30    # approximate nucleus diameter in pixels at 20x magnification
)
n_nuclei = nuc_masks.max()
print(f'  {n_nuclei} nuclei detected')

# --- Whole-cell segmentation ---
# Cellpose v4+ input: 3-channel array where ch0=cytoplasm, ch1=nucleus, ch2=empty
# ER channel is the best cytoplasm marker in Cell Painting - it fills the entire
# cytoplasm in a smooth reticulated network right to the plasma membrane.
# AGP (actin) is stronger at the periphery but patchy in the cell interior.
print('Running Cellpose cyto3 model on ER (ch4) + DNA (ch5) channels...')
cyto_model = models.CellposeModel(model_type='cyto3', gpu=False)

def norm16(img):
    """Normalize image to 16-bit range for Cellpose input."""
    lo, hi = np.percentile(img, 1), np.percentile(img, 99)
    out = np.clip(img, lo, hi).astype(float)
    return ((out - lo) / (hi - lo + 1e-8) * 65535).astype(np.uint16)

three_ch = np.stack([
    norm16(er_crop),                                    # ch0: cytoplasm (ER)
    norm16(dna_crop),                                   # ch1: nucleus (DNA)
    np.zeros_like(dna_crop, dtype=np.uint16)            # ch2: empty
], axis=-1)

cell_masks, cell_flows, _ = cyto_model.eval(
    three_ch,
    diameter=60    # approximate whole-cell diameter in pixels at 20x magnification
)
n_cells = cell_masks.max()
print(f'  {n_cells} cells detected')


In [ ]:
# Visualize segmentation results

def get_boundaries(mask):
    """Return binary boundary image from a labeled mask."""
    dilated = ndimage.binary_dilation(mask > 0)
    eroded  = ndimage.binary_erosion(mask > 0)
    return dilated ^ eroded

cell_bounds = get_boundaries(cell_masks)
nuc_bounds  = get_boundaries(nuc_masks)

# Build overlay on the composite image
overlay = crop_dmso.copy()
overlay[cell_bounds] = [1.0, 1.0, 0.0]   # yellow - cell boundary
overlay[nuc_bounds]  = [1.0, 1.0, 1.0]   # white  - nucleus boundary

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))

axes[0].imshow(contrast_stretch(dna_crop), cmap='Blues')
axes[0].set_title('Ch5: DNA (Hoechst 33342)\nNucleus segmentation input', fontsize=9)
axes[0].axis('off')

axes[1].imshow(nuc_masks, cmap='nipy_spectral', interpolation='nearest')
axes[1].set_title(f'Nucleus masks\n{n_nuclei} nuclei (Cellpose nuclei model)', fontsize=9)
axes[1].axis('off')

axes[2].imshow(cell_masks, cmap='nipy_spectral', interpolation='nearest')
axes[2].set_title(f'Cell masks\n{n_cells} cells (Cellpose cyto3, ER + DNA)', fontsize=9)
axes[2].axis('off')

axes[3].imshow(overlay)
axes[3].set_title('Composite + boundaries\nYellow = cell, White = nucleus', fontsize=9)
axes[3].axis('off')

fig.suptitle(
    'Cellpose Segmentation on Cell Painting Images\n'
    'nuclei model on DNA channel (ch5); cyto3 model on ER (ch4) + DNA (ch5)',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig4_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig4_segmentation.png')


## Part 6: Feature Extraction

With cells and nuclei segmented, we can now measure properties of each object. This is what CellProfiler does at scale across thousands of wells in a real HCA experiment - we are doing it here for one cropped image to make the process concrete.

We extract features in the same families that CellProfiler uses:

- **AreaShape**: geometric properties of the segmented region - area, perimeter, eccentricity, solidity, aspect ratio, form factor
- **Intensity**: brightness statistics per channel inside each region - mean, max, integrated intensity, coefficient of variation
- **Texture**: spatial patterns of intensity - variance, smoothness, energy computed using a local neighborhood approach
- **Granularity**: coarseness of the staining pattern at different spatial scales
- **Radial distribution**: how intensity distributes from the object center outward, capturing whether organelles concentrate near the nucleus or the periphery
- **Neighbors**: how many cells are nearby, distance to nearest neighbors

In a real CellProfiler run, each of these families produces dozens to hundreds of measurements per cell per channel. We compute a representative subset here to show what the numbers mean biologically.

In [ ]:
from skimage.measure import regionprops_table

def extract_areashape(mask):
    """Extract AreaShape features from a labeled mask using regionprops."""
    props = regionprops_table(
        mask,
        properties=[
            'label', 'area', 'perimeter', 'eccentricity',
            'solidity', 'extent', 'major_axis_length',
            'minor_axis_length', 'orientation'
        ]
    )
    df = pd.DataFrame(props)
    # Derived features
    df['aspect_ratio'] = df['major_axis_length'] / (df['minor_axis_length'] + 1e-8)
    df['form_factor']  = (4 * np.pi * df['area']) / (df['perimeter'] ** 2 + 1e-8)
    df['compactness']  = df['perimeter'] ** 2 / (4 * np.pi * df['area'] + 1e-8)
    return df


def extract_intensity(mask, image, channel_name):
    """Extract intensity features for one channel inside each labeled region."""
    props = regionprops_table(
        mask, intensity_image=image.astype(float),
        properties=['label', 'intensity_mean', 'intensity_max', 'intensity_min']
    )
    df = pd.DataFrame(props)
    df.rename(columns={
        'intensity_mean': f'MeanIntensity_{channel_name}',
        'intensity_max':  f'MaxIntensity_{channel_name}',
        'intensity_min':  f'MinIntensity_{channel_name}',
    }, inplace=True)

    # Additional intensity stats computed per object
    ids = np.unique(mask)[1:]   # skip 0 (background)
    int_stds, int_cvs, int_integrated = [], [], []
    for obj_id in ids:
        pixels = image[mask == obj_id].astype(float)
        std = pixels.std()
        mean = pixels.mean()
        int_stds.append(std)
        int_cvs.append(std / (mean + 1e-8))   # coefficient of variation
        int_integrated.append(pixels.sum())

    df[f'StdIntensity_{channel_name}']         = int_stds
    df[f'CV_Intensity_{channel_name}']          = int_cvs
    df[f'IntegratedIntensity_{channel_name}']   = int_integrated
    return df


def extract_texture(mask, image, channel_name, scales=(5, 10, 20)):
    """Extract texture features per object at multiple spatial scales.

    For each scale we compute the variance, smoothness, and energy
    of the intensity within a local neighborhood.
    """
    ids = np.unique(mask)[1:]
    rows = []
    img_f = image.astype(float)

    for obj_id in ids:
        region = (mask == obj_id)
        row = {'label': obj_id}
        pixels_raw = img_f[region]

        for scale in scales:
            # Local standard deviation image at this scale
            local_mean = ndimage.uniform_filter(img_f, size=scale)
            local_sq   = ndimage.uniform_filter(img_f ** 2, size=scale)
            local_var  = np.clip(local_sq - local_mean ** 2, 0, None)
            local_std  = np.sqrt(local_var)

            pixels_std = local_std[region]
            pixels_norm = pixels_raw / (pixels_raw.max() + 1e-8)

            row[f'Texture_Variance_{channel_name}_{scale}']   = pixels_std.var()
            row[f'Texture_Smoothness_{channel_name}_{scale}'] = 1 - 1 / (1 + pixels_std.var())
            # Angular second moment (energy) - high when intensity is uniform
            hist_vals, _ = np.histogram(pixels_norm, bins=16, range=(0, 1), density=True)
            hist_vals = hist_vals / (hist_vals.sum() + 1e-8)
            row[f'Texture_Energy_{channel_name}_{scale}']     = (hist_vals ** 2).sum()

        rows.append(row)

    return pd.DataFrame(rows)


def extract_granularity(mask, image, channel_name, n_scales=5):
    """Extract granularity features per object.

    Granularity measures the coarseness of the staining pattern by
    progressively eroding the image and tracking how much intensity is lost.
    High granularity at small scales means fine-grained punctate staining
    (like mitochondria). High granularity at large scales means coarser texture.
    """
    ids = np.unique(mask)[1:]
    rows = []
    img_f = image.astype(float)

    for obj_id in ids:
        region = (mask == obj_id)
        row = {'label': obj_id}
        base_sum = img_f[region].sum() + 1e-8

        for scale in range(1, n_scales + 1):
            # Erode the image by progressively larger structuring elements
            eroded = ndimage.grey_erosion(img_f, size=scale * 2 + 1)
            eroded_sum = eroded[region].sum()
            # Granularity = fraction of signal removed by erosion
            row[f'Granularity_{scale}_{channel_name}'] = (base_sum - eroded_sum) / base_sum

        rows.append(row)

    return pd.DataFrame(rows)


def extract_radial_distribution(mask, image, channel_name, n_bins=4):
    """Extract radial distribution features per object.

    Measures how intensity distributes from the object center to its edge.
    A compound that causes mitochondria to cluster near the nucleus will
    show high intensity in the inner bins and low intensity in the outer bins.
    """
    ids = np.unique(mask)[1:]
    rows = []
    img_f = image.astype(float)
    dist_map = ndimage.distance_transform_edt(mask > 0)

    for obj_id in ids:
        region = (mask == obj_id)
        row = {'label': obj_id}
        distances = dist_map[region]
        intensities = img_f[region]
        total = intensities.sum() + 1e-8
        max_dist = distances.max() + 1e-8
        norm_dist = distances / max_dist

        for b in range(n_bins):
            lo, hi = b / n_bins, (b + 1) / n_bins
            in_bin = (norm_dist >= lo) & (norm_dist < hi)
            frac = intensities[in_bin].sum() / total
            row[f'RadialDistribution_{b+1}of{n_bins}_{channel_name}'] = frac

        rows.append(row)

    return pd.DataFrame(rows)


def extract_neighbors(mask):
    """Extract neighbor features: nearest neighbor distance and neighbor count."""
    from skimage.measure import regionprops
    props = regionprops(mask)
    centroids = np.array([p.centroid for p in props])
    ids = np.array([p.label for p in props])

    rows = []
    for i, (cid, centroid) in enumerate(zip(ids, centroids)):
        dists = np.sqrt(((centroids - centroid) ** 2).sum(axis=1))
        dists[i] = np.inf   # exclude self
        nearest = dists.min()
        # Count neighbors within 2x the nearest distance
        n_neighbors = (dists < nearest * 2.5).sum()
        rows.append({
            'label': cid,
            'Neighbors_NearestDistance': nearest,
            'Neighbors_Count': n_neighbors
        })

    return pd.DataFrame(rows)


print('Feature extraction functions defined.')

In [ ]:
# Run all feature extraction on the segmented nuclei
# In a real CellProfiler run this is done on every well in the plate
# Here we do it on one cropped field to show what the features mean

print('Extracting AreaShape features from nucleus masks...')
df_shape = extract_areashape(nuc_masks)
print(f'  {len(df_shape)} nuclei x {len(df_shape.columns)} shape features')

print('Extracting Intensity features (all 5 channels)...')
df_int = df_shape[['label']].copy()
for ch_idx, ch_name in enumerate(CHANNEL_NAMES):
    df_ch = extract_intensity(nuc_masks, ch_crop_dmso[ch_idx], ch_name)
    df_int = df_int.merge(df_ch, on='label')
print(f'  {len(df_int.columns) - 1} intensity features per nucleus')

print('Extracting Texture features (DNA, Mito, AGP channels)...')
df_tex = df_shape[['label']].copy()
for ch_name in ['DNA', 'Mito', 'AGP']:   # these names now match the corrected CHANNEL_NAMES
    ch_idx = CHANNEL_NAMES.index(ch_name)
    df_t = extract_texture(nuc_masks, ch_crop_dmso[ch_idx], ch_name)
    df_tex = df_tex.merge(df_t, on='label')
print(f'  {len(df_tex.columns) - 1} texture features per nucleus')

print('Extracting Granularity features (RNA, Mito channels)...')
df_gran = df_shape[['label']].copy()
for ch_name in ['RNA', 'Mito']:   # granularity most meaningful for punctate stains
    ch_idx = CHANNEL_NAMES.index(ch_name)
    df_g = extract_granularity(nuc_masks, ch_crop_dmso[ch_idx], ch_name)
    df_gran = df_gran.merge(df_g, on='label')
print(f'  {len(df_gran.columns) - 1} granularity features per nucleus')

print('Extracting Radial Distribution features (DNA, Mito channels)...')
df_rad = df_shape[['label']].copy()
for ch_name in ['DNA', 'Mito']:
    ch_idx = CHANNEL_NAMES.index(ch_name)   # CHANNEL_NAMES.index('DNA') now correctly returns 4
    df_r = extract_radial_distribution(nuc_masks, ch_crop_dmso[ch_idx], ch_name)
    df_rad = df_rad.merge(df_r, on='label')
print(f'  {len(df_rad.columns) - 1} radial distribution features per nucleus')

print('Extracting Neighbor features...')
df_nbr = extract_neighbors(nuc_masks)
print(f'  {len(df_nbr.columns) - 1} neighbor features per nucleus')

# Merge everything into one per-nucleus feature table
df_all = df_shape.copy()
for df_extra in [df_int, df_tex, df_gran, df_rad, df_nbr]:
    df_all = df_all.merge(df_extra, on='label', how='left')

print(f'\nFinal per-nucleus feature table: {len(df_all)} nuclei x {len(df_all.columns) - 1} features')
df_all.head(3)

## Part 7: Visualize and Interpret the Features

Raw feature tables are hard to interpret. We visualize each feature family to show what the numbers are capturing biologically. In a real HCA experiment, these single-cell distributions are aggregated to one row per well (using mean or median), and those well-level profiles are what go into the UMAP and clustering analysis in Video 6b.

In [ ]:
# AreaShape features - show the distribution of key shape measurements
shape_features = ['area', 'eccentricity', 'solidity', 'form_factor',
                  'aspect_ratio', 'compactness']
shape_labels = ['Area (pixels)', 'Eccentricity\n(0=circle, 1=line)',
                'Solidity\n(1=convex)', 'Form Factor\n(1=circle)',
                'Aspect Ratio\n(1=round)', 'Compactness\n(1=circle)']

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, feat, label in zip(axes, shape_features, shape_labels):
    vals = df_all[feat].dropna()
    ax.hist(vals, bins=20, color='#0D9488', alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(vals.median(), color='#1B2A4A', linewidth=1.8, linestyle='--',
               label=f'median={vals.median():.2f}')
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Number of nuclei', fontsize=9)
    ax.legend(fontsize=8, frameon=False)

fig.suptitle(
    f'AreaShape Features - {n_nuclei} Nuclei from One Field of View\n'
    'In CellProfiler these are computed per cell across thousands of wells',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig5_areashape.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig5_areashape.png')

In [ ]:
# Intensity features - mean intensity per nucleus for each channel
# This shows the typical brightness of each stain across the population

mean_int_cols = [f'MeanIntensity_{ch}' for ch in CHANNEL_NAMES]
colors_rgb = ['#CC44CC', '#FF4444', '#DDCC00', '#44BB44', '#4466FF']   # Mito, AGP, RNA, ER, DNA

fig, axes = plt.subplots(1, 5, figsize=(16, 4))

for ax, col, ch_name, color in zip(axes, mean_int_cols, CHANNEL_NAMES, colors_rgb):
    vals = df_all[col].dropna()
    ax.hist(vals, bins=20, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(vals.median(), color='black', linewidth=1.8, linestyle='--')
    ax.set_title(f'{ch_name}\nmedian={vals.median():.0f}', fontsize=10)
    ax.set_xlabel('Mean intensity (16-bit)', fontsize=8)
    ax.set_ylabel('Nuclei', fontsize=8)

fig.suptitle(
    'Intensity Features - Mean Intensity per Nucleus, per Channel\n'
    'A compound that affects nuclear RNA will shift the RNA distribution',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig6_intensity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig6_intensity.png')

In [ ]:
# Texture features - variance at three scales for DNA (ch5) and Mito (ch1) channels
# High variance = heterogeneous texture (active chromatin, fragmented mito)
# Low variance = smooth, uniform texture (condensed chromatin, fused mito network)

tex_pairs = [
    ('Texture_Variance_DNA_5',  'DNA texture\n(scale 5px)',  '#4466FF'),
    ('Texture_Variance_DNA_10', 'DNA texture\n(scale 10px)', '#2244BB'),
    ('Texture_Variance_DNA_20', 'DNA texture\n(scale 20px)', '#112277'),
    ('Texture_Variance_Mito_5',  'Mito texture\n(scale 5px)',  '#CC44CC'),
    ('Texture_Variance_Mito_10', 'Mito texture\n(scale 10px)', '#993399'),
    ('Texture_Variance_Mito_20', 'Mito texture\n(scale 20px)', '#661166'),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
axes = axes.ravel()

for ax, (col, label, color) in zip(axes, tex_pairs):
    if col not in df_all.columns:
        ax.set_visible(False)
        continue
    vals = df_all[col].dropna()
    ax.hist(vals, bins=20, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(vals.median(), color='black', linewidth=1.8, linestyle='--',
               label=f'median={vals.median():.4f}')
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Nuclei', fontsize=9)
    ax.legend(fontsize=8, frameon=False)

fig.suptitle(
    'Texture Features - Intensity Variance at Multiple Spatial Scales\n'
    'DNA texture reflects chromatin organization; Mito texture reflects mitochondrial network state',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig7_texture.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig7_texture.png')

In [ ]:
# Radial distribution features - where inside the nucleus does each stain concentrate?
# Bin 1 = center, Bin 4 = periphery
# DNA concentrates at center (chromatin around nucleolus)
# Mito distributes differently depending on cell state

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, ch_name, color in zip(axes, ['DNA', 'Mito'], ['#4466FF', '#CC44CC']):
    bin_cols = [f'RadialDistribution_{b}of4_{ch_name}' for b in range(1, 5)]
    bin_cols = [c for c in bin_cols if c in df_all.columns]
    means = [df_all[c].mean() for c in bin_cols]
    stds  = [df_all[c].std()  for c in bin_cols]
    x = np.arange(1, len(means) + 1)

    ax.bar(x, means, color=color, alpha=0.8, edgecolor='white')
    ax.errorbar(x, means, yerr=stds, fmt='none', color='black', capsize=4, linewidth=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Bin {i}\n(center)' if i == 1 else
                        f'Bin {i}\n(edge)' if i == len(means) else
                        f'Bin {i}' for i in x], fontsize=9)
    ax.set_ylabel('Fraction of total intensity', fontsize=10)
    ax.set_title(f'{ch_name} Radial Distribution\n(mean across {n_nuclei} nuclei, error bars = SD)', fontsize=10)

fig.suptitle(
    'Radial Distribution Features - How Intensity Distributes Center to Edge\n'
    'Bin 1 = nuclear center, Bin 4 = nuclear periphery',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig8_radial.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig8_radial.png')

In [ ]:
# Neighbor features and summary overview
# Neighbor distance captures how tightly cells are packed
# EGFR inhibitors reduce cell motility, which changes packing density

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vals_dist = df_all['Neighbors_NearestDistance'].dropna()
axes[0].hist(vals_dist, bins=20, color='#1B2A4A', alpha=0.85, edgecolor='white')
axes[0].axvline(vals_dist.median(), color='#0D9488', linewidth=2, linestyle='--',
                label=f'median = {vals_dist.median():.1f} px')
axes[0].set_xlabel('Distance to nearest neighbor (pixels)', fontsize=10)
axes[0].set_ylabel('Nuclei', fontsize=10)
axes[0].set_title('Nearest Neighbor Distance\nCaptures cell packing density', fontsize=10)
axes[0].legend(fontsize=9, frameon=False)

vals_count = df_all['Neighbors_Count'].dropna()
axes[1].hist(vals_count, bins=range(0, int(vals_count.max()) + 2),
             color='#0D9488', alpha=0.85, edgecolor='white', align='left')
axes[1].axvline(vals_count.median(), color='#1B2A4A', linewidth=2, linestyle='--',
                label=f'median = {vals_count.median():.0f}')
axes[1].set_xlabel('Number of neighbors within 2.5x nearest distance', fontsize=10)
axes[1].set_ylabel('Nuclei', fontsize=10)
axes[1].set_title('Neighbor Count\nCompounds affecting motility shift this distribution', fontsize=10)
axes[1].legend(fontsize=9, frameon=False)

fig.suptitle('Neighbor Features - Spatial Relationships Between Cells', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig9_neighbors.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig9_neighbors.png')

In [ ]:
# Feature heatmap: show all features for each nucleus as a heatmap
# This is what the per-nucleus feature matrix looks like before aggregation
# In CellProfiler output, this table has thousands of rows (one per cell per well)

from sklearn.preprocessing import StandardScaler

# Select a representative subset of features for display
display_features = (
    ['area', 'eccentricity', 'solidity', 'form_factor', 'aspect_ratio'] +
    [f'MeanIntensity_{ch}' for ch in CHANNEL_NAMES] +
    [f'CV_Intensity_{ch}' for ch in CHANNEL_NAMES] +
    ['Texture_Variance_DNA_10', 'Texture_Variance_Mito_10', 'Texture_Variance_AGP_10'] +
    ['Texture_Energy_DNA_10', 'Texture_Energy_Mito_10'] +
    [f'Granularity_{s}_Mito' for s in range(1, 4)] +
    ['RadialDistribution_1of4_DNA', 'RadialDistribution_4of4_DNA',
     'RadialDistribution_1of4_Mito', 'RadialDistribution_4of4_Mito'] +
    ['Neighbors_NearestDistance', 'Neighbors_Count']
)
display_features = [f for f in display_features if f in df_all.columns]

feat_matrix = df_all[display_features].dropna()
scaler = StandardScaler()
feat_scaled = scaler.fit_transform(feat_matrix)

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(feat_scaled.T, aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)

ax.set_yticks(range(len(display_features)))
ax.set_yticklabels(display_features, fontsize=6.5)
ax.set_xlabel(f'Individual nuclei (n = {len(feat_matrix)})', fontsize=10)
ax.set_title(
    'Per-Nucleus Feature Matrix (standardized)\n'
    'Each column = one nucleus. Each row = one feature. Color = z-score.',
    fontsize=11
)
plt.colorbar(im, ax=ax, label='z-score', shrink=0.8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig10_feature_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig10_feature_heatmap.png')
print(f'\nFinal feature table shape: {feat_matrix.shape} (nuclei x features)')

## Part 8: From Single-Cell Features to Well-Level Profiles

The heatmap above shows one row per nucleus. In a real Cell Painting experiment a single 384-well plate produces images from thousands of wells, with 50-200 cells per well. Storing and analyzing millions of single-cell rows is computationally demanding and not always necessary.

The standard practice is to aggregate single-cell measurements to a single well-level profile by taking the median (or mean) across all cells in the well. This gives one row per well - the format used in Video 6b.

Below we compute that aggregation for our one field of view, so you can see what the well-level profile looks like compared to the single-cell matrix.

In [ ]:
# Aggregate to a well-level profile by taking the median across all nuclei
# This mirrors what CellProfiler/pycytominer does for each well in a plate

well_profile = feat_matrix.median(axis=0)
well_profile_df = pd.DataFrame(well_profile).T
well_profile_df.index = ['well_r01c01_f01_dmso']

print(f'Single-cell matrix: {feat_matrix.shape}  (one row per nucleus)')
print(f'Well-level profile: {well_profile_df.shape}  (one row per well)')
print(f'\nExample well-level feature values:')
print(well_profile.head(15).to_string())

# Save for reference
well_profile_df.to_csv(RESULTS_DIR / 'well_profile_example.csv')
feat_matrix.to_csv(RESULTS_DIR / 'single_cell_features_example.csv', index=False)
print(f'\nSaved to {RESULTS_DIR}/: well_profile_example.csv and single_cell_features_example.csv')

In [ ]:
# Final summary figure: the complete pipeline in one image
# Raw image -> segmentation -> single-cell features -> well profile

fig = plt.figure(figsize=(18, 5))
gs = gridspec.GridSpec(1, 5, figure=fig, wspace=0.08)

# Panel 1: composite image
ax0 = fig.add_subplot(gs[0])
ax0.imshow(crop_dmso)
ax0.set_title('1. Raw images\n(5 channels)', fontsize=10, fontweight='bold')
ax0.axis('off')

# Panel 2: segmentation
ax1 = fig.add_subplot(gs[1])
ax1.imshow(overlay)
ax1.set_title(f'2. Segmentation\n({n_cells} cells, Cellpose)', fontsize=10, fontweight='bold')
ax1.axis('off')

# Panel 3: single-cell heatmap (small version)
ax2 = fig.add_subplot(gs[2])
n_show = min(30, len(feat_matrix))
ax2.imshow(feat_scaled[:n_show].T, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax2.set_title(f'3. Per-cell features\n({feat_matrix.shape[1]} features per nucleus)', fontsize=10, fontweight='bold')
ax2.set_xlabel(f'First {n_show} nuclei', fontsize=8)
ax2.set_yticks([])

# Panel 4: aggregation arrow as text panel
ax3 = fig.add_subplot(gs[3])
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.text(0.5, 0.6, 'Aggregate\n(median across\nall cells per well)',
         ha='center', va='center', fontsize=10, fontweight='bold', color='#1B2A4A')
ax3.annotate('', xy=(0.85, 0.5), xytext=(0.15, 0.5),
             arrowprops=dict(arrowstyle='->', color='#0D9488', lw=2.5))
ax3.axis('off')

# Panel 5: well-level profile as bar chart (first 20 features)
ax4 = fig.add_subplot(gs[4])
profile_vals = (well_profile - well_profile.mean()) / (well_profile.std() + 1e-8)
n_bars = min(25, len(profile_vals))
bar_colors = ['#0D9488' if v > 0 else '#94120D' for v in profile_vals.values[:n_bars]]
ax4.barh(range(n_bars), profile_vals.values[:n_bars], color=bar_colors, alpha=0.85)
ax4.axvline(0, color='black', linewidth=0.8)
ax4.set_yticks([])
ax4.set_xlabel('z-score', fontsize=9)
ax4.set_title(f'4. Well-level profile\n(1 row, {len(profile_vals)} features)', fontsize=10, fontweight='bold')

fig.suptitle(
    'The Complete Cell Painting Pipeline: From Images to Morphological Profile\n'
    'Video 6b starts here - loading 10,752 pre-computed well profiles from 1,571 compounds',
    fontsize=11, y=1.04
)
plt.savefig(RESULTS_DIR / 'fig11_pipeline_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/fig11_pipeline_summary.png')

## Summary

This notebook walked through the complete Cell Painting image analysis pipeline:

**Images** - Five 16-bit fluorescence channels capture DNA, ER, RNA, actin/Golgi, and mitochondria simultaneously in each well. Raw pixel values preserve the full dynamic range needed for precise feature extraction.

**Composite** - False-color merging of all five channels produces the iconic Cell Painting image seen in publications. Each color encodes a different cellular compartment.

**Segmentation** - Cellpose identifies every nucleus using the DNA channel, then expands to the full cell boundary guided by the AGP (actin) channel. The output is a labeled mask array.

**Feature extraction** - For each segmented cell we computed:
- AreaShape: geometric properties (area, eccentricity, form factor, compactness)
- Intensity: brightness statistics per channel (mean, CV, integrated intensity)
- Texture: spatial pattern statistics at multiple scales (variance, energy, smoothness)
- Granularity: coarseness of the staining pattern at increasing erosion scales
- Radial distribution: how intensity distributes from cell center to edge
- Neighbors: intercellular spacing and local density

**Aggregation** - Single-cell measurements are aggregated to one row per well using the median. That one row is the morphological profile.

In Video 6b we load 10,752 such profiles from 1,571 compounds across 6 doses, merge them with mechanism-of-action annotations, and use PCA, UMAP, and hierarchical clustering to ask whether compounds with the same mechanism produce similar morphological fingerprints.